[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/54_rope_2d_solution.ipynb)

# Solution: 2D Rotary Position Embedding (2D RoPE)

Reference solution.


In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch


In [ ]:
# ✅ SOLUTION

def _apply_1d_rope(x, positions):
    dim = x.shape[-1]
    half = dim // 2
    inv_freq = 1.0 / (10000 ** (torch.arange(0, half, 2, device=x.device, dtype=x.dtype) / half))
    angles = positions.to(dtype=x.dtype).unsqueeze(-1) * inv_freq.unsqueeze(0)
    cos = torch.cos(angles).repeat_interleave(2, dim=-1).unsqueeze(0)
    sin = torch.sin(angles).repeat_interleave(2, dim=-1).unsqueeze(0)
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]
    x_rot = torch.stack([-x_odd, x_even], dim=-1).reshape_as(x)
    return x * cos + x_rot * sin


def apply_2d_rope(q, k, height, width):
    seq_len = height * width
    assert q.shape[1] == seq_len and k.shape[1] == seq_len
    dim = q.shape[-1]
    assert dim % 4 == 0, 'D must be divisible by 4'
    half = dim // 2
    rows = torch.arange(height, device=q.device).unsqueeze(1).expand(height, width).reshape(-1)
    cols = torch.arange(width, device=q.device).unsqueeze(0).expand(height, width).reshape(-1)
    q_row, q_col = q[..., :half], q[..., half:]
    k_row, k_col = k[..., :half], k[..., half:]
    q_rot = torch.cat([_apply_1d_rope(q_row, rows), _apply_1d_rope(q_col, cols)], dim=-1)
    k_rot = torch.cat([_apply_1d_rope(k_row, rows), _apply_1d_rope(k_col, cols)], dim=-1)
    return q_rot, k_rot


In [ ]:
q = torch.randn(1, 16, 32)
k = torch.randn(1, 16, 32)
qr, kr = apply_2d_rope(q, k, height=4, width=4)
print('Q shape:', qr.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))


In [ ]:
from torch_judge import check
check('rope_2d')
